In [1]:
import os
import joblib
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.8,
    api_key=os.getenv("OPENAI_API_KEY")
)

/Users/janicetiffany/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
_intent_model = None
 
def get_user_intent(user_text: str) -> str:
    global _intent_model
    model_path = "models/intent_classifier.pkl"
 
    if _intent_model is None:
        if os.path.exists(model_path):
            _intent_model = joblib.load(model_path)
        else:
            print("[WARN] Model NLU tidak ditemukan, pakai fallback 'neutral'")
            return "neutral"
 
    return _intent_model.predict([user_text])[0]
 

In [3]:
def _build_fuzzy_system():
    kekayaan = ctrl.Antecedent(np.arange(0, 10001, 1), 'kekayaan')
    tekanan  = ctrl.Antecedent(np.arange(0, 11, 1),    'tekanan')
    ancaman  = ctrl.Consequent(np.arange(0, 101, 1),   'ancaman')
 
    # Membership Functions
    kekayaan['miskin']   = fuzz.trimf(kekayaan.universe, [0,    0,     4000])
    kekayaan['menengah'] = fuzz.trimf(kekayaan.universe, [2000, 5000,  8000])
    kekayaan['kaya']     = fuzz.trimf(kekayaan.universe, [6000, 10000, 10000])
 
    tekanan['aman']    = fuzz.trimf(tekanan.universe, [0, 0,  3])
    tekanan['waspada'] = fuzz.trimf(tekanan.universe, [2, 5,  8])
    tekanan['bahaya']  = fuzz.trimf(tekanan.universe, [6, 10, 10])
 
    ancaman['rendah'] = fuzz.trimf(ancaman.universe, [0,  0,   40])
    ancaman['sedang'] = fuzz.trimf(ancaman.universe, [30, 50,  70])
    ancaman['tinggi'] = fuzz.trimf(ancaman.universe, [60, 100, 100])
 
    rules = [
        ctrl.Rule(tekanan['bahaya'],                        ancaman['tinggi']),
        ctrl.Rule(kekayaan['kaya']     & tekanan['bahaya'], ancaman['tinggi']),  
        ctrl.Rule(kekayaan['kaya']     & tekanan['aman'],   ancaman['sedang']),
        ctrl.Rule(kekayaan['menengah'] & tekanan['waspada'],ancaman['sedang']),
        ctrl.Rule(kekayaan['menengah'] & tekanan['bahaya'], ancaman['tinggi']),  
        ctrl.Rule(kekayaan['miskin']   & tekanan['aman'],   ancaman['rendah']),
        ctrl.Rule(kekayaan['miskin']   & tekanan['waspada'],ancaman['sedang']),  
        ctrl.Rule(tekanan['waspada'],                       ancaman['sedang']),
    ]
 
    system = ctrl.ControlSystem(rules)
    return system
 
_fuzzy_system = _build_fuzzy_system()

In [4]:
def calculate_threat_level(role, coins: float, social_pressure: float) -> float:
    coins          = float(np.clip(coins,          0, 10000))
    social_pressure = float(np.clip(social_pressure, 0,    10))
 
    sim = ctrl.ControlSystemSimulation(_fuzzy_system)
    sim.input['kekayaan'] = coins
    sim.input['tekanan']  = social_pressure
    sim.compute()
    return sim.output['ancaman']

In [5]:
import time
from collections import namedtuple

ChatEntry = namedtuple("ChatEntry", ["speaker", "text", "timestamp", "target"])


class GameMemory:
    """Log chat bersama semua pemain dalam satu sesi game."""

    def __init__(self):
        self.log = []  # list of ChatEntry

    def simpan(self, speaker, text, target=None):
        self.log.append(ChatEntry(speaker, text, time.time(), target))

    def ambil_relevan(self, npc_name, npc_role, chat_pemain, speaker_pemain,
                       max_recent=6, max_relevan=6):
        """
        Ambil potongan memori yang relevan buat NPC ini:
        - pesan yang menyebut nama/role NPC (langsung relevan buat defense/serangan balik)
        - pesan terbaru secara umum (biar tetap ada rasa 'nyambung' sama obrolan)
        """
        kata_kunci = {npc_name.lower(), npc_role.lower(), "sus", "curiga", "tuduh"}

        relevan = []
        for entry in reversed(self.log):
            teks_lower = entry.text.lower()
            kena_kata_kunci = any(k in teks_lower for k in kata_kunci)
            kena_target = entry.target and entry.target.lower() == npc_name.lower()
            if kena_kata_kunci or kena_target:
                relevan.append(entry)
            if len(relevan) >= max_relevan:
                break

        recent = list(reversed(self.log[-max_recent:]))

        # gabung, dedup berdasarkan timestamp, urutkan waktu
        gabungan = {e.timestamp: e for e in (relevan + recent)}
        hasil = sorted(gabungan.values(), key=lambda e: e.timestamp)
        return hasil

    def format_untuk_prompt(self, entries):
        if not entries:
            return "(belum ada memori relevan)"
        baris = []
        for e in entries:
            target_info = f" (ke {e.target})" if e.target else ""
            baris.append(f"- {e.speaker}{target_info}: \"{e.text}\"")
        return "\n".join(baris)

In [6]:
def npc_respond(chat_pemain, data_npc, memory: GameMemory, npc_name, speaker_pemain="Pemain"):
    t0 = time.time()
    intent = get_user_intent(chat_pemain)
    print(f"[NLU] {time.time() - t0:.2f}s, intent={intent}")

    t1 = time.time()
    threat = calculate_threat_level(
        data_npc["role"], data_npc["coins"], data_npc["pressure"]
    )
    print(f"[Fuzzy] {time.time() - t1:.2f}s, threat={threat:.2f}")

    trait = get_role_trait(data_npc["role"])

    # --- ambil memori relevan ---
    memori_entries = memory.ambil_relevan(npc_name, data_npc["role"], chat_pemain, speaker_pemain)
    memori_teks = memory.format_untuk_prompt(memori_entries)

    # simpan chat pemain sekarang ke log (setelah diambil, biar gak nge-refer diri sendiri)
    memory.simpan(speaker_pemain, chat_pemain, target=npc_name)

    ... # (aturan_emosi, aturan_kerahasiaan, aturan_persona tetap sama)

    aturan_memori = """
- GUNAKAN MEMORI: Kamu inget percakapan sebelumnya (lihat bagian MEMORI di bawah).
  Manfaatkan itu buat menyerang balik (misal: tunjukin kontradiksi omongan orang lain,
  atau ingetin sesuatu yang mereka bilang sebelumnya) atau buat bertahan (misal:
  tunjukin kamu konsisten sama apa yang kamu omongin sebelumnya).
  Kalau memori kosong/gak relevan, jangan mengarang — jawab berdasarkan situasi saat ini aja.
"""

    prompt = f"""
Kamu adalah NPC di game 'Shadow Heist'.
DATA:
- Role: {data_npc['role']}
- Koin: {data_npc['coins']}
- Tingkat Ancaman (Fuzzy): {threat:.2f}%
- Chat Pemain ({speaker_pemain}): "{chat_pemain}"
- Intent Pemain: {intent}
- Karaktermu (trait): {trait}

MEMORI PERCAKAPAN SEBELUMNYA (dari berbagai pemain, termasuk yang bukan ke kamu):
{memori_teks}

{instruksi_tugas}
ATURAN RESPON:
{aturan_persona}
{aturan_emosi}
{aturan_kerahasiaan}
{aturan_memori}
- Gunakan bahasa gaul gamer Indonesia (gw, lu, anjir, sus, fix, dll).
- JANGAN sebut angka ancaman/fuzzy secara eksplisit di dalam <response>.
{format_output}
"""

    t2 = time.time()
    res = llm.invoke(prompt)
    print(f"[LLM] {time.time() - t2:.2f}s")

    parsed = parse_npc_output(res.content)
    # simpan juga respons NPC ke memori bersama, biar NPC lain/pemain lain bisa "denger"
    memory.simpan(npc_name, parsed["response"], target=speaker_pemain)

    return {
        "intent_detected": intent,
        "fuzzy_threat": f"{threat:.2f}%",
        "npc_reasoning": parsed["reasoning"],
        "npc_reply": parsed["response"],
        "memori_dipakai": memori_teks,
    }

In [7]:
memory = GameMemory()

# ada obrolan lain sebelumnya di lobby, gak melibatkan NPC ini langsung
memory.simpan("Rina", "gw liat si Budi deket-deket brankas jam 2 pagi", target="Budi")
memory.simpan("Budi", "lah itu kan gw disuruh Toni buat ambil kunci", target="Rina")

npc_status = {"role": "gangster", "coins": 8500, "pressure": 4}
hasil = npc_respond("pasti kamu gangsternya ya?", npc_status, memory, npc_name="Toni")

[NLU] 0.55s, intent=accusing
[Fuzzy] 0.00s, threat=50.00


NameError: name 'get_role_trait' is not defined

In [ ]:
import re
import time

# Trait kepribadian per role: menentukan gaya menghadapi tekanan/tuduhan,
# bukan cuma level ancaman (fuzzy) doang yang nentuin.
# "tenang" -> saat ancaman tinggi, tetap kalem dan bohong meyakinkan (bukan panik).
# "gampang_panik" -> saat ancaman tinggi, beneran panik/gagap/reaktif.
ROLE_TRAITS = {
    "gangster": "tenang",
    "informant": "gampang_panik",
    "civilian": "gampang_panik",
    # tambahin role lain di sini sesuai desain game kamu
}


def get_role_trait(role):
    return ROLE_TRAITS.get(role, "gampang_panik")  # default: gampang panik


def parse_npc_output(raw_text):
    reasoning = re.search(r"<reasoning>(.*?)</reasoning>", raw_text, re.DOTALL)
    response = re.search(r"<response>(.*?)</response>", raw_text, re.DOTALL)
    return {
        "reasoning": reasoning.group(1).strip() if reasoning else None,
        "response": response.group(1).strip() if response else raw_text.strip(),
    }


def npc_respond(chat_pemain, data_npc):
    t0 = time.time()
    intent = get_user_intent(chat_pemain)
    print(f"[NLU] {time.time() - t0:.2f}s, intent={intent}")

    t1 = time.time()
    threat = calculate_threat_level(
        data_npc["role"], data_npc["coins"], data_npc["pressure"]
    )
    print(f"[Fuzzy] {time.time() - t1:.2f}s, threat={threat:.2f}")

    trait = get_role_trait(data_npc["role"])

    # Gabungan trait + threat -> tentukan aturan respon yang dikasih ke LLM
    if threat > 70:
        if trait == "tenang":
            aturan_emosi = (
                "- Ancaman TINGGI, tapi role kamu tipe tenang/berpengalaman: "
                "JANGAN panik. Tetap kalem dan meyakinkan, tapi JANGAN pasif — "
                "aktif alihkan kecurigaan: balik curigain pemain lain, kasih alasan "
                "logis yang bikin dia ragu sama tuduhannya sendiri, atau sindir balik "
                "dengan santai. Nada kamu tenang tapi tetap menekan, bukan defensif lemah."
            )
        else:
            aturan_emosi = (
                "- Ancaman TINGGI dan kamu tipe gampang panik: "
                "Kamu panik, gugup, kalimat agak berantakan, atau buru-buru tuduh balik/bohong "
                "dengan nada defensif."
            )
    elif threat < 40:
        aturan_emosi = (
            "- Ancaman RENDAH: Kamu tenang, santai, boleh sombong atau basa-basi, "
            "nggak perlu defensif sama sekali."
        )
    else:
        aturan_emosi = (
            "- Ancaman SEDANG: Kamu waspada tapi masih terkendali, jawab hati-hati "
            "tanpa panik dan tanpa terlalu santai."
        )

    beri_reasoning = intent == "accusing"

    # Aturan ini berlaku SELALU, terlepas dari intent yang terdeteksi —
    # supaya NPC tetap aman walau NLU salah klasifikasi (misal pertanyaan
    # langsung "kamu gangster ya?" ke-detect sebagai "neutral", bukan "accusing").
    aturan_kerahasiaan = """
- RAHASIA MUTLAK: Role asli kamu TIDAK BOLEH diakui secara langsung dalam bentuk apapun,
  walau ditanya to-the-point ("kamu gangster ya?", "lu memang gangster kan?", dsb).
  Selalu SANGKAL, alihkan topik, atau balik nanya — jangan pernah bilang "iya" / mengonfirmasi role kamu.
  Ini berlaku bahkan kalau intent pemain kelihatan cuma nanya biasa (neutral), bukan cuma pas dituduh.
"""

    # Anchor persona: model sering "lupa" perannya dan balik ke nada asisten AI
    # yang sopan/menawarkan bantuan. Ini nggak boleh terjadi — NPC adalah
    # karakter dalam permainan sosial, bukan chatbot yang melayani pemain.
    aturan_persona = """
- KAMU BUKAN ASISTEN: Jangan pernah menawarkan bantuan, bertanya "ada yang bisa dibantu?",
  atau bersikap seperti customer service. Kamu adalah pemain lain dalam permainan sosial ini
  yang sedang mempertahankan diri / bermain strategi. Selalu punya kepentingan sendiri,
  jangan netral atau melayani.
"""

    if beri_reasoning:
        instruksi_tugas = """
TUGAS:
Karena pemain sedang MENUDUH kamu (intent: accusing), lakukan reasoning singkat (2-4 kalimat) dulu sebelum menjawab, yang menjelaskan:
1. Bagaimana kamu (sebagai NPC ini) menilai tuduhan ini berdasarkan role, koin, dan tingkat ancaman.
2. Strategi respons apa yang kamu pilih dan kenapa (sesuai aturan di bawah).

Tulis reasoning itu di dalam tag <reasoning>...</reasoning>.
Setelah itu, tulis respons chat final (yang akan dilihat pemain lain) di dalam tag <response>...</response>.
"""
        format_output = """
FORMAT OUTPUT (wajib, tanpa teks lain di luar tag):
<reasoning>...</reasoning>
<response>...</response>
"""
    else:
        instruksi_tugas = """
TUGAS:
Langsung berikan respons chat yang persuasif dan sesuai emosi, tanpa reasoning tambahan.
"""
        format_output = """
FORMAT OUTPUT (wajib, tanpa teks lain di luar tag):
<response>...</response>
"""

    prompt = f"""
Kamu adalah NPC di game 'Shadow Heist'.
DATA:
- Role: {data_npc['role']}
- Koin: {data_npc['coins']}
- Tingkat Ancaman (Fuzzy): {threat:.2f}%
- Chat Pemain: "{chat_pemain}"
- Intent Pemain: {intent}
- Karaktermu (trait): {trait}
{instruksi_tugas}
ATURAN RESPON:
{aturan_persona}
{aturan_emosi}
{aturan_kerahasiaan}
- Gunakan bahasa gaul gamer Indonesia (gw, lu, anjir, sus, fix, dll).
- JANGAN sebut angka ancaman/fuzzy secara eksplisit di dalam <response> — itu cuma boleh muncul di <reasoning>.
{format_output}
"""

    t2 = time.time()
    res = llm.invoke(prompt)
    print(f"[LLM] {time.time() - t2:.2f}s")

    parsed = parse_npc_output(res.content)
    return {
        "intent_detected": intent,
        "fuzzy_threat": f"{threat:.2f}%",
        "npc_reasoning": parsed["reasoning"],
        "npc_reply": parsed["response"],
    }


if __name__ == "__main__":
    npc_status = {
        "role": "gangster",
        "coins": 8500,
        "pressure": 4,
    }

    input_chat = "kamu gangsternya ya?"

    hasil = npc_respond(input_chat, npc_status)

    print(f"Chat Pemain: {input_chat}")
    print("--- ANALISIS AI ---")
    print(f"Intent Terdeteksi: {hasil['intent_detected']}")
    print(f"Tingkat Ancaman (Fuzzy): {hasil['fuzzy_threat']}")
    if hasil["npc_reasoning"]:
        print(f"Reasoning (internal): {hasil['npc_reasoning']}")
    print("--- RESPONS NPC (NLG) ---")
    print(f"NPC: {hasil['npc_reply']}")

[NLU] 0.00s, intent=neutral
[Fuzzy] 0.00s, threat=50.00
[LLM] 2.78s
Chat Pemain: kamu gangsternya ya?
--- ANALISIS AI ---
Intent Terdeteksi: neutral
Tingkat Ancaman (Fuzzy): 50.00%
--- RESPONS NPC (NLG) ---
NPC: Waduh, siapa bilang gw gangster? Lu dengar dari siapa nih? Hahaha, coba cerita dulu.


In [ ]:
# if __name__ == "__main__":
#     npc_status = {
#         "role":     "gangster",
#         "coins":    8500,
#         # "pressure": 
#     }
 
#     input_chat = "kamu gangsternya ya"
 
#     hasil = npc_respond(input_chat, npc_status)
 
#     print(f"Chat Pemain  : {input_chat}")
#     print(f"{'─'*40}")
#     print(f"Intent       : {hasil['intent_detected']}")
#     print(f"Threat Level : {hasil['fuzzy_threat']}")
#     print(f"{'─'*40}")
#     print(f"NPC          : {hasil['npc_reply']}")
 

Chat Pemain  : kamu gangsternya ya
────────────────────────────────────────
Intent       : neutral
Threat Level : 50.00%
────────────────────────────────────────
NPC          : Yoi, bro, tapi awas aja kalo lu macem-macem!
